# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE # RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES. import   kagglehub kagglehub.login() VBox(children=(HTML(value='<center> <img\nsrc=https://www.kaggle.com/static/images/site-logo.png\nalt=\'Kaggle... import   kagglehub kagglehub.login() # IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES, # THEN FEEL FREE TO DELETE THIS CELL. #   NOTE : THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON # ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR # NOTEBOOK. titanic_path   =   kagglehub.competition_download( 'titanic' ) print ( 'Data source import complete.' ) VBox(children=(HTML(value='<center> <img\nsrc=https://www.kaggle.com/static/images/site-logo.png\nalt=\'Kaggle... --------------------------------------------------------------------------- UnauthenticatedError                      Traceback (most recent call last) /tmp/ipykernel_3884/3779395555.py in <cell line: 0>() 8 # NOTEBOOK. 9 ---> 10 titanic_path = kagglehub.competition_download('titanic') 11 12 print('Data source import complete.') /usr/local/lib/python3.12/dist-packages/kagglehub/competition.py in competition_download(handle, path, force_download, output_dir) 28     h = parse_competition_handle(handle) 29     logger.info(f"Downloading competition: {h.to_url()} ...", extra={**EXTRA_CONSOLE_BLOCK}) ---> 30     path, _ = registry.competition_resolver( 31         h, 32         path, /usr/local/lib/python3.12/dist-packages/kagglehub/registry.py in __call__(self, *args, **kwargs) 26         for impl in reversed(self._impls): 27             if impl.is_supported(*args, **kwargs): ---> 28                 return impl(*args, **kwargs) 29             else:

--- End of Page 1 ---

30                 fails.append(type(impl).__name__) /usr/local/lib/python3.12/dist-packages/kagglehub/resolver.py in __call__(self, handle, path, force_download, output_dir) 33             Some cases where version number might be missing: Competition datasource, API-based models. 34         """ ---> 35         path, version = self._resolve( 36             handle, 37             path, /usr/local/lib/python3.12/dist-packages/kagglehub/http_resolver.py in _resolve(self, h, path, force_download, output_dir) 53                 if cached_path: 54                     return cached_path, None ---> 55                 raise UnauthenticatedError() 56 57             out_path = cache.get_path(h, path) UnauthenticatedError: User is not authenticated import   os kaggle_dir   =   '/root/.kaggle' kaggle_json_path   =   os.path.join(kaggle_dir,   'kaggle.json' ) if   os.path.exists(kaggle_json_path): print ( f'kaggle.json found at:   { kaggle_json_path } ' ) # Optionally, you can also print its content for verification (be careful with sensitive info) # with open(kaggle_json_path, 'r') as f: #     print('Content of kaggle.json:') #     print(f.read()) else : print ( f'kaggle.json NOT found at:   { kaggle_json_path } ' ) print ( 'Please ensure the kaggle.json file is uploaded to this location.' ) # Also list the directory content to see all files in .kaggle print ( f'Contents of   { kaggle_dir } :' ) ! ls   - la {kaggle_dir} kaggle.json NOT found at: /root/.kaggle/kaggle.json Please ensure the kaggle.json file is uploaded to this location. Contents of /root/.kaggle: ls: cannot access '/root/.kaggle': No such file or directory

--- End of Page 2 ---

Python for Data 25: Chi-Squared Tests  back to index Last lesson we introduced the framework of statistical hypothesis testing and the t-test for investigating differences between numeric variables. In this lesson, we turn our attention to a common statistical test for categorical variables: the chi- squared test.  Chi-Squared Goodness-Of-Fit Test  In our study of t-tests, we introduced the one-way t-test to check whether a sample mean differs from the an expected (population) mean. The chi-squared goodness-of-fit test is an analog of the one-way t-test for categorical variables: it tests whether the distribution of sample categorical data matches an expected distribution. For example, you could use a chi-squared goodness-of-fit test to check whether the race demographics of members at your church or school match that of the entire U.S. population or whether the computer browser preferences of your friends match those of Internet uses as a whole. When working with categorical data, the values themselves aren’t of much use for statistical testing because categories like “male”, “female,” and “other” have no mathematical meaning. Tests dealing with categorical variables are based on variable counts instead of the actual value of the variables themselves. Let’s generate some fake demographic data for U.S. and Minnesota and walk through the chi-square goodness of fit test to check whether they are different:  import   numpy   as   np import   pandas   as   pd import   scipy.stats   as   stats national   =   pd.DataFrame([ "white" ] * 100000   +   [ "hispanic" ] * 60000   +\ [ "black" ] * 50000   +   [ "asian" ] * 15000   +   [ "other" ] * 35000 ) minnesota   =   pd.DataFrame([ "white" ] * 600   +   [ "hispanic" ] * 300   +   \ [ "black" ] * 250   + [ "asian" ] * 75   +   [ "other" ] * 150 ) national_table   =   pd.crosstab(index = national[ 0 ], columns = "count" ) minnesota_table   =   pd.crosstab(index = minnesota[ 0 ], columns = "count" ) print (   "National" ) print (national_table)

--- End of Page 3 ---

print ( " " ) print (   "Minnesota" ) print (minnesota_table) National col_0      count 0 asian      15000 black      50000 hispanic   60000 other      35000 white     100000 Minnesota col_0     count 0 asian        75 black       250 hispanic    300 other       150 white       600  Chi-squared tests are based on the so-called chi-squared statistic. You calculate the chi-squared statistic with the following formula: $$ sum(\frac{(observed-expected)^2}{expected}) $$ In the formula, observed is the actual observed count for each category and expected is the expected count based on the distribution of the population for the corresponding category. Let’s calculate the chi-squared statistic for our data to illustrate:  observed   =   minnesota_table national_ratios   =   national_table / len (national)   # Get population ratios expected   =   national_ratios   *   len (minnesota)   # Get expected counts chi_squared_stat   =   (((observed - expected) ** 2 ) / expected). sum () print (chi_squared_stat) col_0 count    18.194805 dtype: float64  Note: The chi-squared test assumes none of the expected counts are less than 5.

--- End of Page 4 ---

Similar to the t-test where we compared the t-test statistic to a critical value based on the t-distribution to determine whether the result is significant, in the chi-square test we compare the chi-square test statistic to a critical value based on the   chi-square distribution . The scipy library shorthand for the chi-square distribution is chi2. Let’s use this knowledge to find the critical value for 95% confidence level and check the p-value of our result:  crit   =   stats.chi2.ppf(q   =   0.95 ,   # Find the critical value for 95% confidence* df   =   4 )   # Df = number of variable categories - 1 print ( "Critical value" ) print (crit) p_value   =   1   -   stats.chi2.cdf(x = chi_squared_stat,   # Find the p-value df = 4 ) print ( "P value" ) print (p_value) Critical value 9.487729036781154 P value [0.00113047]  Since our chi-squared statistic exceeds the critical value, we’d reject the null hypothesis that the two distributions are the same. You can carry out a chi-squared goodness-of-fit test automatically using the scipy function scipy.stats.chisquare():  stats.chisquare(f_obs =   observed,   # Array of observed counts f_exp =   expected)   # Array of expected counts Power_divergenceResult(statistic=array([18.19480519]), pvalue=array([0.00113047]))  The test results agree with the values we calculated above.  Chi-Squared Test of Independence  Independence   is a key concept in probability that describes a situation where knowing the value of one variable tells you nothing about the value of another. For instance, the month you were born probably doesn’t tell you anything about which web browser you use, so we’d expect birth month and browser preference to be independent. On the other hand, your month of birth might be related to whether you excelled at sports in school, so month of birth and sports performance might not be independent. The chi-squared test of independence tests whether two categorical variables are

--- End of Page 5 ---

independent. The test of independence is commonly used to determine whether variables like education, political views and other preferences vary based on demographic factors like gender, race and religion. Let’s generate some fake voter polling data and perform a test of independence:  np.random.seed( 10 ) # Sample data randomly at fixed probabilities voter_race   =   np.random.choice(a =   [ "asian" , "black" , "hispanic" , "other" , "white" ], p   =   [ 0.05 ,   0.15   , 0.25 ,   0.05 ,   0.5 ], size = 1000 ) # Sample data randomly at fixed probabilities voter_party   =   np.random.choice(a =   [ "democrat" , "independent" , "republican" ], p   =   [ 0.4 ,   0.2 ,   0.4 ], size = 1000 ) voters   =   pd.DataFrame({ "race" :voter_race, "party" :voter_party}) voter_tab   =   pd.crosstab(voters.race, voters.party, margins   =   True ) voter_tab.columns   =   [ "democrat" , "independent" , "republican" , "row_totals" ] voter_tab.index   =   [ "asian" , "black" , "hispanic" , "other" , "white" , "col_totals" ] observed   =   voter_tab.iloc[ 0 : 5 , 0 : 3 ]   # Get table without totals for later use voter_tab <div>   democrat   independent   republican   row_totals asian   21   7   32   60  black   65   25   64   154  hispanic   107   50   94   251  other   15   8   15   38  white   189   96   212   497  col_totals   397   186   417   1000  <div class="colab-df-buttons"> <button class="colab-df-convert" onclick="convertToInteractive('df-811f5b0f-4f15-437b-a362-0c1c73005add')" title="Convert this dataframe to an interactive table." style="display:none;">

--- End of Page 6 ---

<style> .colab-df-generate { background-color: #E8F0FE; border: none; border-radius: 50%; cursor: pointer; display: none; fill: #1967D2; height: 32px; padding: 0 0 0 0; width: 32px; } .colab-df-generate:hover { background-color: #E2EBFA; box-shadow: 0px 1px 2px rgba(60, 64, 67, 0.3), 0px 1px 3px 1px rgba(60, 64, 67, 0.15); fill: #174EA6; } [theme=dark] .colab-df-generate { background-color: #3B4455; fill: #D2E3FC; } [theme=dark] .colab-df-generate:hover { background-color: #434B5C; box-shadow: 0px 1px 3px 1px rgba(0, 0, 0, 0.15); filter: drop-shadow(0px 1px 2px rgba(0, 0, 0, 0.3));

--- End of Page 7 ---

fill: #FFFFFF; } </style> <button class="colab-df-generate" onclick="generateWithVariable('voter_tab')" title="Generate code using this dataframe." style="display:none;"> </div>  Note that we did not use the race data to inform our generation of the party data so the variables are independent. For a test of independence, we use the same chi-squared formula that we used for the goodness-of-fit test. The main difference is we have to calculate the expected counts of each cell in a 2-dimensional table instead of a 1-dimensional table. To get the expected count for a cell, multiply the row total for that cell by the column total for that cell and then divide by the total number of observations. We can quickly get the expected counts for all cells in the table by taking the row totals and column totals of the table, performing an outer product on them with the np.outer() function and dividing by the number of observations:  expected   =   np.outer(voter_tab[ "row_totals" ][ 0 : 5 ], voter_tab.loc[ "col_totals" ][ 0 : 3 ])   /   1000 expected   =   pd.DataFrame(expected) expected.columns   =   [ "democrat" , "independent" , "republican" ] expected.index   =   [ "asian" , "black" , "hispanic" , "other" , "white" ] expected <div>   democrat   independent   republican asian   23.820   11.160   25.020  black   61.138   28.644   64.218  hispanic   99.647   46.686   104.667  other   15.086   7.068   15.846  white   197.309   92.442   207.249  <div class="colab-df-buttons"> <button class="colab-df-convert" onclick="convertToInteractive('df-03ebc27d-baf1-44f0-958e-dd02318c01fd')" title="Convert this dataframe to an interactive table." style="display:none;">

--- End of Page 8 ---

<style> .colab-df-generate { background-color: #E8F0FE; border: none; border-radius: 50%; cursor: pointer; display: none; fill: #1967D2; height: 32px; padding: 0 0 0 0; width: 32px; } .colab-df-generate:hover { background-color: #E2EBFA; box-shadow: 0px 1px 2px rgba(60, 64, 67, 0.3), 0px 1px 3px 1px rgba(60, 64, 67, 0.15); fill: #174EA6; } [theme=dark] .colab-df-generate { background-color: #3B4455; fill: #D2E3FC; } [theme=dark] .colab-df-generate:hover {

--- End of Page 9 ---

background-color: #434B5C; box-shadow: 0px 1px 3px 1px rgba(0, 0, 0, 0.15); filter: drop-shadow(0px 1px 2px rgba(0, 0, 0, 0.3)); fill: #FFFFFF; } </style> <button class="colab-df-generate" onclick="generateWithVariable('expected')" title="Generate code using this dataframe." style="display:none;"> </div>  Now we can follow the same steps we took before to calculate the chi-square statistic, the critical value and the p-value:  chi_squared_stat   =   (((observed - expected) ** 2 ) / expected). sum (). sum () print (chi_squared_stat) 7.169321280162059  Note: We call .sum() twice: once to get the column sums and a second time to add the column sums together, returning the sum of the entire 2D table.  crit   =   stats.chi2.ppf(q   =   0.95 ,   # Find the critical value for 95% confidence* df   =   8 )   # * print ( "Critical value" ) print (crit) p_value   =   1   -   stats.chi2.cdf(x = chi_squared_stat,   # Find the p-value df = 8 ) print ( "P value" ) print (p_value) Critical value 15.50731305586545 P value 0.518479392948842  Note: The degrees of freedom for a test of independence equals the product of the number of categories in each variable minus 1. In this case we have a 5x3 table so df = 4x2 = 8.  As with the goodness-of-fit test, we can use scipy to conduct a test of independence quickly. Use stats.chi2_contingency() function to conduct a test of

--- End of Page 10 ---

independence automatically given a frequency table of observed counts:  stats.chi2_contingency(observed =   observed) Chi2ContingencyResult(statistic=np.float64(7.169321280162059), pvalue=np.float64(0.518479392948842), dof=8, expected_freq=array([[ 23.82 ,  11.16 ,  25.02 ], [ 61.138,  28.644,  64.218], [ 99.647,  46.686, 104.667], [ 15.086,   7.068,  15.846], [197.309,  92.442, 207.249]]))  The output shows the chi-square statistic, the p-value and the degrees of freedom followed by the expected counts. As expected, given the high p-value, the test result does not detect a significant relationship between the variables.  Wrap Up  Chi-squared tests provide a way to investigate differences in the distributions of categorical variables with the same categories and the dependence between categorical variables. In the next lesson, we’ll learn about a third statistical inference test, the analysis of variance, that lets us compare several sample means at the same time.  Next Lesson:   Python for Data 26: ANOVA  back to index  Q1.When do we go for Chi square test ?   Ans. You go for a Chi-squared test when you are working with categorical variables. Specifically, you use it for two main purposes: Chi-squared Goodness-of-Fit Test: To see if the observed distribution of a single categorical variable matches an expected distribution. Chi-squared Test of Independence: To determine if there is a significant relationship or association between two (or more) categorical variables.  Q2.Do we use Chi square test for independence ? Explain How ?   Ans. Yes, absolutely! The Chi-squared Test of Independence is specifically designed to determine if there is a statistically significant association (or dependence) between two categorical variables. Here’s how it works:

--- End of Page 11 ---

Formulating Hypotheses: You start with two hypotheses: Null Hypothesis (H0): The two categorical variables are independent (there is no association between them). Alternative Hypothesis (H1): The two categorical variables are dependent (there is an association between them). Observed Frequencies: You collect data and arrange it into a contingency table (also known as a cross-tabulation). This table shows the observed counts for each combination of categories of your two variables. In the notebook, the voter_tab DataFrame (cell_id: i_Zn04yVAzRX) serves as an example of an observed frequency table, showing the counts of voters by race and party. Expected Frequencies: Under the assumption that the null hypothesis is true (i.e., the variables are independent), you calculate the expected count for each cell in the contingency table. The formula for an expected count for a cell is: (row total * column total) / grand total. The notebook demonstrates this calculation in cell_id: fb1W8FMdAzRX to generate the expected DataFrame. Chi-squared Statistic Calculation: You then calculate the Chi-squared test statistic using the formula: $$ \sum \frac{(Observed - Expected)^2}{Expected} $$ This sum is performed over all cells in the contingency table. A larger difference between observed and expected counts leads to a larger chi-squared statistic. The notebook calculates this in cell_id: jQe7JrY_AzRY. Degrees of Freedom (df): For a test of independence, the degrees of freedom are calculated as (number of rows - 1) * (number of columns - 1). This is explicitly stated and used in cell_id: DfWWRGqZAzRY. P-value and Conclusion: Finally, you compare your calculated chi-squared statistic to a chi-squared distribution with the appropriate degrees of freedom to obtain a p-value. If the p-value is less than your chosen significance level (e.g., 0.05), you reject the null hypothesis and conclude that there is a significant association between the two variables. Otherwise, you fail to reject the null hypothesis, suggesting no significant association. The notebook shows this in cell_id: xOch0zI- AzRY and cell_id: 8Ind67bJAzRY. The scipy.stats.chi2_contingency() function (cell_id: 8Ind67bJAzRY) automates all these steps, providing the statistic, p-value, degrees of freedom, and expected frequencies directly.  Q3Explain role of Critical value and p-value in Chi square test.   Ans.In a Chi- squared test, both the critical value and the p-value are crucial for making a decision about your hypothesis: Critical Value Definition: The critical value is a threshold value obtained from the appropriate statistical distribution (in this case, the Chi-squared distribution). It defines the boundary of the rejection region. How it’s used: If your calculated Chi- squared test statistic (which measures the discrepancy between observed and expected frequencies) is greater than the critical value, it falls into the rejection

--- End of Page 12 ---

region. This means the observed difference is considered statistically significant, and you reject the null hypothesis. Factors: The critical value depends on the chosen significance level (alpha, commonly 0.05 for 95% confidence) and the degrees of freedom of your test. In the notebook: In cell_id: Lp1ej6laAzRX and cell_id: xOch0zI-AzRY, the critical value was calculated using stats.chi2.ppf(q = 0.95, df = ...). For the goodness-of-fit test, crit was 9.4877, and for the independence test, crit was 15.5073. P-value Definition: The p-value is the probability of observing a test statistic as extreme as, or more extreme than, the one calculated from your sample data, assuming that the null hypothesis is true. How it’s used: You compare the p-value to your chosen significance level (alpha). If the p-value is less than or equal to alpha (e.g., p <= 0.05), you reject the null hypothesis, indicating that the observed data is unlikely if the null hypothesis were true. If the p-value is greater than alpha, you fail to reject the null hypothesis. Interpretation: A small p-value (e.g., 0.01) suggests strong evidence against the null hypothesis, while a large p-value (e.g., 0.60) suggests weak evidence against the null hypothesis. In the notebook: In cell_id: Lp1ej6laAzRX and cell_id: xOch0zI-AzRY, the p-value was calculated using 1 - stats.chi2.cdf(x=chi_squared_stat, df=...). For the goodness-of-fit test, the p-value was 0.0011, which is less than 0.05, leading to the rejection of the null hypothesis. For the independence test, the p-value was 0.5185, which is greater than 0.05, leading to a failure to reject the null hypothesis. What can I help you build?  Q4Can we use chi square test to show existence of correlation between 2 features in a dataset ? How will you interpret test result to indicate correlation presence.   Ans. Yes, absolutely! While the term “correlation” is often used more broadly, in strict statistical terms, a Chi-squared test of independence is used to assess the association or dependence between two categorical features (variables) in a dataset, rather than a linear “correlation” which typically applies to continuous variables. Here’s how you interpret the test results to indicate the presence of an association: Formulate Hypotheses (Recap): Null Hypothesis (H0): The two categorical variables are independent (there is no association between them). Alternative Hypothesis (H1): The two categorical variables are dependent (there is an association between them). Calculate the Chi-squared Statistic: This statistic quantifies the difference between the observed frequencies in your contingency table and the frequencies you would expect if the variables were truly independent. A larger chi-squared statistic indicates a greater deviation from independence. Determine the P-value: The p-value is the probability of observing a chi-squared statistic as extreme as, or more extreme than, the one you calculated, assuming the null hypothesis (independence) is true. The notebook calculates this for the voter data in cell_id: xOch0zI-AzRY, resulting in a p-value of 0.5185.

--- End of Page 13 ---

Interpret the P-value: If P-value < Significance Level (alpha, typically 0.05): You reject the null hypothesis. This means there is statistically significant evidence to suggest that the two categorical features are dependent (or associated). In layman’s terms, knowing the category of one feature gives you information about the likely category of the other feature. The smaller the p-value, the stronger the evidence against independence. If P-value >= Significance Level (alpha): You fail to reject the null hypothesis. This means there is not enough statistically significant evidence to conclude that the two categorical features are dependent. You would infer that they are independent (or not associated). Example from the Notebook: In the notebook’s Chi-squared test of independence for voter race and party (cell_id: xOch0zI-AzRY and cell_id: 8Ind67bJAzRY): The calculated p-value was approximately 0.5185. Since 0.5185 is much greater than the common significance level of 0.05, We fail to reject the null hypothesis. Conclusion: This indicates that there is no statistically significant association between voter race and voter party preference in the generated sample data. The variables are considered independent. In essence, a significant Chi-squared test result (low p-value) suggests that there’s a relationship between the categories of your two variables, meaning they are not independent.  Q5What is degrees of freedom and how to calculate it while performing chi square test.   Ans. Degrees of freedom (df) is a fundamental concept in statistics that refers to the number of independent values or quantities that can be varied in a statistical calculation without violating any constraints. Think of it as the number of values in a final calculation of a statistic that are free to vary. If you have a set of numbers that must sum to a certain total, then all but one of those numbers can vary freely; the last number is determined by the total. In essence, it’s the number of pieces of information that can vary independently. How to calculate degrees of freedom for Chi-squared tests: For Chi-squared Goodness-of-Fit Test: Calculation: df = (number of categories) - 1 Explanation: In this test, you are comparing observed counts to expected counts across a single categorical variable with multiple categories. Once you know the total number of observations, all but one category’s count can vary freely; the last one is determined by the total. Example from notebook: In cell_id: Lp1ej6laAzRX, for the Minnesota demographic data, there are 5 race categories. So, df = 5 - 1 = 4. For Chi-squared Test of Independence: Calculation: df = (number of rows - 1) * (number of columns - 1) Explanation: This test involves a contingency table with multiple rows and columns (representing the categories of two categorical variables). For this 2-dimensional table, the degrees of freedom are calculated by multiplying one less than the number of rows by one less than the number of columns. This accounts for the constraints imposed by the row totals, column totals, and the grand total. Example from

--- End of Page 14 ---

notebook: In cell_id: DfWWRGqZAzRY, for the voter race and party data, you have a 5x3 table (5 race categories, 3 party categories). So, df = (5 - 1) * (3 - 1) = 4 * 2 = 8.